In [ ]:
# Step 4 — Radial distribution test (RUWE-low vs RUWE-high) and CDF plot
#
# Purpose
#   - Load a final member catalog (CSV) produced by Step 1.
#   - Compute projected angular radius from a reproducible cluster center.
#   - Split the sample into RUWE-low and RUWE-high groups.
#   - Compare radial distributions using a two-sample Kolmogorov–Smirnov test.
#   - Plot empirical cumulative distributions (CDFs).
#
# Notes
#   - RUWE is treated as an astrometric-excess proxy, not a deterministic binary label.
#   - The radius is computed using a small-angle approximation on the tangent plane.

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
@dataclass
class SegregationConfig:
    cluster_name: str
    input_members_csv: str

    # Column names
    ra_col: str = "ra"
    dec_col: str = "dec"
    ruwe_col: str = "ruwe"

    # RUWE thresholds (leave a gap to reduce overlap)
    ruwe_single_max: float = 1.1
    ruwe_binary_min: float = 1.2

    # Center definition
    # - "mean": center is mean RA and mean Dec of all members (reproducible)
    center_method: str = "mean"  # "mean" only in this implementation

    # Plot controls
    xlim_arcmin: Optional[float] = 60.0
    bins: int = 1000
    figsize: Tuple[float, float] = (10.0, 7.0)

    # Output (optional)
    save_path: Optional[str] = None  # e.g., "M67_segregation_cdf.png"
    dpi: int = 300


# ---------------------------------------------------------------------
# Core functionality
# ---------------------------------------------------------------------
def load_members(path: str) -> pd.DataFrame:
    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f"Members file not found: {path_obj.resolve()}")
    return pd.read_csv(path_obj)


def compute_center(df: pd.DataFrame, cfg: SegregationConfig) -> Tuple[float, float]:
    if cfg.center_method != "mean":
        raise ValueError("Only center_method='mean' is supported in this notebook.")
    center_ra = float(df[cfg.ra_col].mean())
    center_dec = float(df[cfg.dec_col].mean())
    return center_ra, center_dec


def add_projected_radius(df: pd.DataFrame, cfg: SegregationConfig) -> pd.DataFrame:
    """
    Compute projected separation using a small-angle tangent-plane approximation:
      r_deg = sqrt( (ΔRA * cos(dec0))^2 + (ΔDec)^2 )
    and convert to arcminutes.
    """
    df = df.copy()

    center_ra, center_dec = compute_center(df, cfg)

    d_ra = (df[cfg.ra_col].values - center_ra) * np.cos(np.radians(center_dec))
    d_dec = (df[cfg.dec_col].values - center_dec)
    r_deg = np.sqrt(d_ra**2 + d_dec**2)

    df["radius_deg"] = r_deg
    df["radius_arcmin"] = r_deg * 60.0
    df.attrs["center_ra"] = center_ra
    df.attrs["center_dec"] = center_dec
    return df


def split_by_ruwe(df: pd.DataFrame, cfg: SegregationConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    single = df[df[cfg.ruwe_col] < cfg.ruwe_single_max].copy()
    binary = df[df[cfg.ruwe_col] > cfg.ruwe_binary_min].copy()
    return single, binary


def ks_test(single: pd.DataFrame, binary: pd.DataFrame) -> Tuple[float, float]:
    if len(single) == 0 or len(binary) == 0:
        return np.nan, np.nan
    stat, pval = ks_2samp(single["radius_arcmin"].values, binary["radius_arcmin"].values)
    return float(stat), float(pval)


def plot_cdf(single: pd.DataFrame, binary: pd.DataFrame, cfg: SegregationConfig, p_value: float) -> None:
    plt.figure(figsize=cfg.figsize)

    plt.hist(
        single["radius_arcmin"].values,
        bins=cfg.bins,
        density=True,
        cumulative=True,
        histtype="step",
        linewidth=2,
        label=f"RUWE < {cfg.ruwe_single_max:.2f}",
    )
    plt.hist(
        binary["radius_arcmin"].values,
        bins=cfg.bins,
        density=True,
        cumulative=True,
        histtype="step",
        linewidth=2,
        label=f"RUWE > {cfg.ruwe_binary_min:.2f}",
    )

    title_p = "NA" if np.isnan(p_value) else f"{p_value:.2e}"
    plt.title(f"{cfg.cluster_name}: radial CDF comparison (KS p-value = {title_p})")
    plt.xlabel("Projected distance from center (arcmin)")
    plt.ylabel("Cumulative fraction")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="lower right")

    if cfg.xlim_arcmin is not None:
        plt.xlim(0, cfg.xlim_arcmin)

    plt.tight_layout()

    if cfg.save_path is not None:
        out_path = Path(cfg.save_path)
        plt.savefig(out_path, dpi=cfg.dpi)
        print(f"Saved CDF figure: {out_path.resolve()}")

    plt.show()


def main(cfg: SegregationConfig) -> None:
    df = load_members(cfg.input_members_csv)

    # Coerce required columns to numeric and drop NaNs
    for col in (cfg.ra_col, cfg.dec_col, cfg.ruwe_col):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=[cfg.ra_col, cfg.dec_col, cfg.ruwe_col]).copy()

    df = add_projected_radius(df, cfg)
    center_ra = df.attrs.get("center_ra", np.nan)
    center_dec = df.attrs.get("center_dec", np.nan)

    single, binary = split_by_ruwe(df, cfg)
    stat, pval = ks_test(single, binary)

    print("Sample sizes")
    print(f"  Total members: {len(df)}")
    print(f"  RUWE-low  (RUWE < {cfg.ruwe_single_max:.2f}): {len(single)}")
    print(f"  RUWE-high (RUWE > {cfg.ruwe_binary_min:.2f}): {len(binary)}")
    print("")
    print("Cluster center (mean of members)")
    print(f"  RA0  = {center_ra:.6f} deg")
    print(f"  Dec0 = {center_dec:.6f} deg")
    print("")
    print("Two-sample KS test (radius_arcmin)")
    if np.isnan(pval):
        print("  Insufficient data in one or both subsamples; KS test not computed.")
    else:
        print(f"  KS statistic = {stat:.4f}")
        print(f"  p-value      = {pval:.2e}")

    plot_cdf(single, binary, cfg, pval)


# ---------------------------------------------------------------------
# User configuration (edit here)
# ---------------------------------------------------------------------
cfg = SegregationConfig(
    cluster_name="M67",
    input_members_csv="M67_Members_Final.csv",
    ruwe_single_max=1.1,
    ruwe_binary_min=1.2,
    xlim_arcmin=60.0,
    bins=1000,
    save_path=None,  # e.g., "M67_segregation_cdf.png"
)

main(cfg)
